# 6. Split the data by farm

**Open:** restart R and save the supplied script as `scripts/my-06-split-farms.R`.

**Input:** `outputs/checkpoints/msc-prepared.rds` from Chapter 5.

## 6.1 Choose the unit to hold out

Our question is prediction for a **farm not used to learn the rule**. All records from a selected farm must therefore stay together.

```{admonition} Invented data — split three farms
:class: note

Here are the six practice records from Chapter 4.

![Training keeps both rounds from farms A and B; testing keeps both rounds from farm C.](../assets/farm-groups.svg)
```

A row-wise split can put records from the same farm on both sides. It then answers a different prediction question.

**Check:** seeing a second sample from farm A is not the same as seeing a farm for the first time. Which situation matches our question?

## 6.2 Load and make the real split

**Study data.** Run the supplied {span .script-block}`BLOCK · load-prepared` block to reload the saved table. After a restart, an empty Environment is expected. This block reads the disk checkpoint with `readRDS()`, then takes its `data` item as `usable`. If the file is missing, complete Chapter 5 first:

In [1]:
if (!file.exists("outputs/checkpoints/msc-prepared.rds")) stop("Run Chapter 5 first to create msc-prepared.rds.")
prepared <- readRDS("outputs/checkpoints/msc-prepared.rds")
usable <- prepared$data
stopifnot(nrow(usable) == 525)

**The decision:** hold out 20% of the 102 farms, rounding up to 21. Use the supplied seed, 2026, to repeat this selection. Keep these choices fixed while learning the procedure.

The code draws farm IDs first. Every record whose farm is on that list goes into the test set, and the remaining records go into training.

Run the supplied {span .script-block}`BLOCK · split-by-farm` block unchanged. Expand its code if you want to inspect the random selection and row assignment:

In [2]:
# kind sets how R generates random numbers. sample.kind sets how sample() draws.
# normal.kind applies to draws from a normal distribution.
RNGkind(kind = "Mersenne-Twister", normal.kind = "Inversion", sample.kind = "Rejection")
set.seed(2026)  # Set the starting point so sample() can repeat the draw.
farm_ids <- sort(unique(usable$FarmID), method = "radix")  # Sort one ID per farm.
test_farms <- sample(farm_ids, size = ceiling(0.20 * length(farm_ids)))
# %in% checks if a farm is on the test list. ! reverses that check for training.
train <- usable[!usable$FarmID %in% test_farms, ]
test <- usable[usable$FarmID %in% test_farms, ]
c(train_rows = nrow(train), test_rows = nrow(test),
  train_farms = length(unique(train$FarmID)),
  test_farms = length(unique(test$FarmID)))

train_rows   test_rows train_farms  test_farms 
        420         105          81          21

## 6.3 Read the checks as evidence

| Check | Supplied split |
|---|---:|
| Training farms | 81 |
| Test farms | 21 |
| Training records | 420 |
| Test records | 105 |

Record counts need not divide into exactly 80% and 20%, because farms contribute different numbers of records.

Run {span .script-block}`BLOCK · check-farm-overlap` to inspect the actual shared labels:

In [3]:
overlap <- intersect(unique(train$FarmID), unique(test$FarmID))
overlap
stopifnot(length(overlap) == 0)
stopifnot(nrow(train) + nrow(test) == nrow(usable))

character(0)

**Expected:** `character(0)`, an empty list of farm labels. If you see any label, repair the split before fitting a model. This addresses farm overlap; it does not establish independence of every record or performance in a new region.

## 6.4 Save a starting point for evaluation

Run the supplied {span .script-block}`BLOCK · save-split` block:

In [4]:
split_audit <- rbind(prepared$audit,
  data.frame(check = "train/test farm overlap", value = length(overlap)))
saveRDS(list(train = train, test = test, farm_ids = farm_ids,
             test_farms = test_farms, audit = split_audit), "outputs/checkpoints/farm-split.rds")
file.exists("outputs/checkpoints/farm-split.rds")

[1] TRUE

**Find:** `outputs/checkpoints/farm-split.rds`. You can now restart R; Chapter 7 reloads the saved split.

**Explain before moving on:** why both samples from the same farm share a role, and why an empty farm intersection is a better check than row counts alone.

**Next:** [learn and evaluate a baseline](07-baseline.ipynb).